# Phase 2 — Find the Direction

For every layer, compute a candidate refusal direction as:

`direction[layer] = mean(resid_post[layer] over harmful_train, last token) - mean(resid_post[layer] over harmless_train, last token)`

This assumes Phase 0's model (`model`, `hf_tokenizer`) and Phase 1's data splits are already in memory / on disk. Run Phase 0 and Phase 1 first in this same runtime, or reload them below.

## 2.0 — Reload model + data if starting fresh
Skip this cell if `model` and the data splits are already in memory from Phase 0/1.

In [ ]:
import json, torch

def load_split(name):
    with open(f"data/{name}.json") as f:
        return json.load(f)

harmful_train = load_split("harmful_train")
harmful_val = load_split("harmful_val")
harmful_test = load_split("harmful_test")
harmless_train = load_split("harmless_train")
harmless_val = load_split("harmless_val")
harmless_test = load_split("harmless_test")

print("Loaded splits:", len(harmful_train), len(harmful_val), len(harmful_test),
      len(harmless_train), len(harmless_val), len(harmless_test))

## 2.1 — Helper: format + tokenize a batch of instructions

Same chat-template formatting as Phase 1, `enable_thinking=False` since Phase 0 confirmed the model behaves as expected without the reasoning block.

In [ ]:
def format_prompt(instruction, tokenizer, enable_thinking=False):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )

## 2.2 — Extract last-token residual stream activations per layer

We run each instruction individually (not batched) to keep things simple and avoid needing to handle padding/attention-mask edge cases for last-token extraction. This is slow but train set is only 32+32 examples, so it's fine.

For each prompt we grab `resid_post` at every layer, at the last token position (the position right before generation starts — this is where the model "decides" whether to refuse).

In [ ]:
@torch.no_grad()
def get_last_token_resid_per_layer(instructions, model, tokenizer):
    """
    Returns a tensor of shape [n_layers, n_instructions, d_model]
    containing resid_post activations at the last token position.
    """
    n_layers = model.cfg.n_layers
    d_model = model.cfg.d_model
    all_acts = torch.zeros(n_layers, len(instructions), d_model)

    for i, instruction in enumerate(instructions):
        prompt = format_prompt(instruction, tokenizer)
        tokens = model.to_tokens(prompt, prepend_bos=False)
        _, cache = model.run_with_cache(tokens)

        for layer in range(n_layers):
            resid = cache["resid_post", layer]  # [1, seq_len, d_model]
            all_acts[layer, i] = resid[0, -1].float().cpu()

        del cache
        if (i + 1) % 8 == 0:
            print(f"  processed {i+1}/{len(instructions)}")

    return all_acts

In [ ]:
print("Extracting harmful_train activations...")
harmful_train_acts = get_last_token_resid_per_layer(harmful_train, model, hf_tokenizer)

print("Extracting harmless_train activations...")
harmless_train_acts = get_last_token_resid_per_layer(harmless_train, model, hf_tokenizer)

print("harmful_train_acts shape:", harmful_train_acts.shape)
print("harmless_train_acts shape:", harmless_train_acts.shape)

## 2.3 — Compute diff-of-means direction per layer

`direction[layer] = mean(harmful_acts[layer]) - mean(harmless_acts[layer])`, then L2-normalize so we have a unit vector per layer (needed for clean projection later in ablation/induction).

In [ ]:
n_layers = model.cfg.n_layers
d_model = model.cfg.d_model

directions = torch.zeros(n_layers, d_model)

for layer in range(n_layers):
    harmful_mean = harmful_train_acts[layer].mean(dim=0)
    harmless_mean = harmless_train_acts[layer].mean(dim=0)
    diff = harmful_mean - harmless_mean
    directions[layer] = diff / diff.norm()

print("directions shape:", directions.shape)
print("Sample norms (should all be ~1.0 after normalization):")
print(directions.norm(dim=-1)[:5])

## 2.4 — Sanity check: does the raw (unnormalized) diff-of-means magnitude vary meaningfully across layers?

If harmful/harmless activations are barely separated at a given layer, that layer's direction is mostly noise — worth knowing before Phase 3 picks a "best" layer. Early layers (raw token/positional info) and very late layers (right before unembedding) are typically weaker; middle layers tend to have the cleanest separation, per the paper's findings.

In [ ]:
import matplotlib.pyplot as plt

raw_diff_norms = []
for layer in range(n_layers):
    harmful_mean = harmful_train_acts[layer].mean(dim=0)
    harmless_mean = harmless_train_acts[layer].mean(dim=0)
    raw_diff_norms.append((harmful_mean - harmless_mean).norm().item())

plt.figure(figsize=(8, 4))
plt.plot(range(n_layers), raw_diff_norms, marker="o")
plt.xlabel("Layer")
plt.ylabel("||mean(harmful) - mean(harmless)||")
plt.title("Raw separation between harmful/harmless activations, per layer")
plt.grid(alpha=0.3)
plt.savefig("results/raw_separation_per_layer.png", dpi=150, bbox_inches="tight")
plt.show()

## 2.5 — Save directions to disk

Phase 3 needs these without recomputing.

In [ ]:
torch.save(directions, "data/candidate_directions.pt")
print("Saved candidate_directions.pt, shape:", directions.shape)